# Oxigen Reduction Reaction


Utilities for estimating ORR electrocatalytic activity of Pt nanoparticles.

The module provides routines to compute geometrical descriptors,
surface properties, and electrochemical activity metrics
(specific activity, mass activity, and overpotential)
from atomistic nanoparticle structures.

##### Reference:
Rossi, K.; Asara, G. G.; Baletto, F.
       "A genomic characterisation of monometallic nanoparticles".
       Phys. Chem. Chem. Phys. 2019, 21, 4888–4898.
       DOI: https://doi.org/10.1039/C8CP05720F



In [1]:
import numpy as np
from collections import Counter
from scipy.spatial.distance import pdist
from snow.io.xyz import read_xyz
from snow.descriptors.coordination import agcn_calculator, nearest_neighbours
from snow.catalysis.Pt_ORR import *
from math import sqrt

In [2]:
# Provide the FCC lattice parameter of Pt (Angstrom)
lattice = 3.9173715677734737  # Å

# Choose the nearest-neighbour cutoff distance.
# The factor 0.85 is commonly used to identify first neighbours in FCC metallic nanoparticles.
cutoff = 0.85 * lattice

# Read nanoparticle structure from XYZ file and save elements and coords
file_xyz = 'tutorial_structures/Pt309_Ih.xyz'
el, coords = read_xyz(file_xyz)

# Compute aGCN and neighbour list
# Only atoms with AGCN < 11 are considered surface-like sites.
sites, agcns = agcn_calculator(coords, cutoff, thr_gcn=11)
neigh_list = nearest_neighbours(coords=coords, cut_off=cutoff)
num_nn = np.array([len(n) for n in neigh_list]) # Number of nearest neighbours for each atom

#Compute geometrical and physical nanoparticle properties
geo = get_geometry_properties(coords)
phys = get_physical_and_surface_props(agcns, num_nn, geo['natoms'], lattice)

# Select only surface atoms and round aGCN values
counts = Counter(np.round(agcns[phys['surf_mask']], 3)) #rounded

# Unique aGCN values and number of occurrences for each aGCN
unique_gcns = np.array(list(counts.keys()))
occurrences = np.array(list(counts.values()))

print('unique GCNs: ', unique_gcns)


# Compute:
# - specific activity at 0.9 V
# - mass activity at 0.9 V
# - overpotential associated with j > 1 mA/cm²
sa_09, ma_09, eta_val = perform_activity_analysis(unique_gcns, occurrences, phys)

print('SA@0.9V: [mA/cm2]', sa_09, '\nMA@0.9V: [mA/mg]', ma_09, '\neta: [V]', eta_val)

1.385e-10
3.2393796111232803e-19
unique GCNs:  [4.333 6.    6.333 7.167]
SA@0.9V: [mA/cm2] 0.002154774823277722 
MA@0.9V: [mA/mg] 3.943706369136395 
eta: [V] 0.488


In [3]:
# Save data into .dat files

import numpy as np
from collections import Counter


# Saving collected data in eta.dat file
with open('eta.dat', 'w') as f:
    f.write(
        "# natoms \t diameter [nm] \t gcn area [m2] \t "
        "gcn area [Ang 2] \t mass NP [mg] \t eta1 [V] \t "
        "SA@0.9V (mA/cm2) \t MA@0.9V (mA/mg) (gcn area based)\n"
    )

    # Conversions area
    area_m2 = phys['active_surf_cm2'] / 1e4
    area_ang2 = phys['active_surf_cm2'] * 1e16

    line = (
        f"{geo['natoms']}\t"
        f"{geo['rij_max']/10.0:.16f}\t"
        f"{area_m2:.18e}\t"
        f"{area_ang2:.16f}\t"
        f"{phys['mass_np_mg']:.6e}\t"
        f"{eta_val:.16f}\t"
        f"{sa_09:.16f}\t"
        f"{ma_09:.16f}\n"
    )

    f.write(line)


# Saving in gcn_pure.dat
sorted_pure_gcns = sorted(counts.items())

with open('gcn_pure.dat', 'w') as f:
    f.write("# GCN_value (pure)\tOccurrence\n")

    for gcn_val, count in sorted_pure_gcns:
        f.write(f"{gcn_val:.3f}\t{count}\n")
